# ImmunoStruct: Run inference on new data

This notebook runs **ImmunoStruct** immunogenicity prediction on a single peptide-MHC pair.  
You provide the **MHC allele** and **peptide sequence**; we run remote AlphaFold2 folding and then inference.

**Run in Google Colab:**  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KrishnaswamyLab/ImmunoStruct/blob/main/demo/infer_new_data.ipynb)

*Enable a GPU in Colab (Runtime → Change runtime type → GPU) for faster structure prediction.*

Colab setup: install deps and clone repo (skip if running locally)

In [ ]:
import os
import sys
import subprocess

def in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if in_colab():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    # Match Colab's preinstalled jaxlib (0.7.x) so "import jax" works; run every time
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-warn-conflicts", "--upgrade",
        "dm-haiku", "jax[cuda12_pip]==0.7.2",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"])
    # Remove broken TF kernel so TensorFlow loads (run every time; see ColabFold notebook)
    import glob
    for f in glob.glob("/usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so"):
        try: os.remove(f)
        except Exception: pass
    # ColabFold + AlphaFold2 (use install that works on Colab Python 3.12; can take several minutes)
    if not os.path.isfile("/content/COLABFOLD_READY"):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
            "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"])
        open("/content/COLABFOLD_READY", "w").close()
    if not os.path.exists("/content/ImmunoStruct"):
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/KrishnaswamyLab/ImmunoStruct.git", "/content/ImmunoStruct"])
    sys.path.insert(0, "/content/ImmunoStruct/immunostruct")
else:
    # Local: if needed, add path to immunostruct (e.g. parent of demo/)
    pass

print("Setup done.")

Download model weights from Hugging Face.

In [ ]:
from huggingface_hub import hf_hub_download
import os

REPO_ID = "ChenLiu1996/ImmunoStruct"
REPO_TYPE = "model"

os.makedirs("weights", exist_ok=True)
iedb_path = hf_hub_download(repo_id=REPO_ID, filename="IEDB_model_seed1.pt", repo_type=REPO_TYPE, local_dir="weights")
cedar_path = hf_hub_download(repo_id=REPO_ID, filename="CEDAR_model_seed2.pt", repo_type=REPO_TYPE, local_dir="weights")
print("IEDB model:", iedb_path)
print("CEDAR model:", cedar_path)

Provide peptide-MHC data to run inference on.

In [ ]:
peptide = "YTDQISKYA"
allele = "HLA-A*01:01"
# See `data/HLA_allele_sequences.csv` for the allele-sequence mapping for some common MHCs.
MHC_sequence = "SHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQKMEPRAPWIEQEGPEYWDQETRNMKAHSQTDRANLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAVHAAEQRRVYLEGRCVDGLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLR"

**Step 1. MSA generation + AlphaFold2 folding**  
(1) Download AlphaFold2 params (~3GB, first time only).  
(2) Run remote MSA via MMseqs2.  
(3) Run AlphaFold2 to predict the peptide–MHC structure.  
Output: PDB(s) in `structure/<jobname>/`.

In [ ]:
import re
import os
from pathlib import Path
# Force JAX to fully load first to avoid "partially initialized module 'jax'" circular import
import jax
import jax.numpy as jnp  # noqa: F401

# Import from installed ColabFold *before* adding preprocessing to path (preprocessing/colabfold.py would shadow the package)
from colabfold.download import download_alphafold_params

if "ImmunoStruct" in os.getcwd() or os.path.exists("/content/ImmunoStruct"):
    _prep = "/content/ImmunoStruct/immunostruct/preprocessing" if os.path.exists("/content/ImmunoStruct") else os.path.join(os.getcwd(), "immunostruct", "preprocessing")
    if _prep not in __import__("sys").path:
        __import__("sys").path.insert(0, _prep)
import colabfold_alphafold as cf_af

# --- 1) Download AlphaFold2 params (first time only, ~3GB) ---
params_dir = Path("/content/alphafold_params")
params_dir.mkdir(parents=True, exist_ok=True)
if not list(params_dir.glob("params_model_*_ptm.npz")):
    print("Downloading AlphaFold2 params (first time only, ~3GB)...")
    download_alphafold_params("alphafold2_ptm", params_dir)
params_loc = str(params_dir)

# --- 2) MSA: prepare inputs and run remote MSA ---
full_sequence_folding = f"{MHC_sequence}:{peptide}"
full_sequence_name = f"{allele}:{peptide}"
full_sequence_name = re.sub(r'\W+', '_', full_sequence_name)

output_dir = os.path.join("structure", full_sequence_name)
os.makedirs(output_dir, exist_ok=True)

I = cf_af.prep_inputs(full_sequence_folding, full_sequence_name, "1:1", output_dir=output_dir, clean=False)
print(f"Prepared inputs for {full_sequence_name}.", flush=True)

cf_af.prep_msa(
    I,
    msa_method="mmseqs2",
    add_custom_msa=False,
    msa_format="fas",
    pair_mode="unpaired",
    pair_cov=50,
    pair_qid=20,
    TMP_DIR="/tmp/",
)
print(f"Prepared MSA for {full_sequence_name}.", flush=True)

# --- 3) Folding: build features and run AlphaFold2 ---
feature_dict = cf_af.prep_feats(I, clean=False)
opt = {
    "N": len(feature_dict["msa"]),
    "L": len(feature_dict["residue_index"]),
    "use_ptm": True,
    "use_turbo": True,
    "num_relax": None,
    "max_recycles": 3,
    "tol": 0.0,
    "num_ensemble": 1,
    "max_msa_clusters": 256,
    "max_extra_msa": 512,
    "is_training": False,
}
runner = cf_af.prep_model_runner(opt, params_loc=params_loc)
# step4: num_models=1, num_samples=1, subsample_msa=True, rank_by="pLDDT", show_images=False
_, _ = cf_af.run_alphafold(
    feature_dict, opt, runner,
    num_models=1, num_samples=1, subsample_msa=True,
    rank_by="pLDDT", show_images=False, params_loc=params_loc,
)
print(f"Step 1 done. Structure(s) in: {I['output_dir']}", flush=True)

